## Daily Challenge: Pinecone Serverless Reranking in Action

### Part 1: Load Documents & Execute Reranking Model

#### 1. Install Pinecone libraries

Run this command in your notebook cell (note the `!` for notebook execution). You'll need the client package to interact with Pinecone's API and the notebook helper to simplify authentication in environments like Colab.

In [ ]:
!pip install -U pinecone==6.0.1 pinecone-notebooks

#### 2. Authenticate with Pinecone

Run this code block exactly as shown. It will prompt for your API key if not already set. Securely providing your API key lets the client connect to your Pinecone project without hard-coding secrets in your script.

In [ ]:
import os
if not os.environ.get("PINECONE_API_KEY"):
    from pinecone_notebooks.colab import Authenticate
    Authenticate()

#### 3. Instantiate the Pinecone client

Copy this code exactly. The modern Pinecone client auto-detects your environment. The client (`pc`) is your entry point for all Pinecone operations—creating indexes, querying, and reranking.

In [ ]:
from pinecone import Pinecone
api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=api_key)

#### 4. Define your query & documents

Replace each `...` with actual text documents that mix references to Apple (company) and apple (fruit). You need a small set of documents to test the reranker’s ability to distinguish between different contexts of the same word.

In [ ]:
query = "Tell me about Apple's products"
documents = [
    "The crisp red apple is a delicious and healthy fruit, often eaten as a snack.", # Add a document about apple fruit
    "Apple Inc. designs, manufactures, and markets smartphones, personal computers, tablets, wearables, and accessories worldwide.", # Add a document about Apple company products
    "Apple pie is a classic dessert made with baked apples, cinnamon, and a pastry crust.", # Add another fruit-related document
    "The new Apple Vision Pro headset was announced, promising an immersive mixed reality experience.", # Add another company-related document
    "Many people enjoy drinking apple juice in the morning as a refreshing beverage." # Add one more document (your choice)
]

#### 5. Call the reranker

Fill in `top_n` with how many top results you want returned (e.g., 3). `top_n` limits the number of reranked results, so you only retrieve the most relevant documents.

In [ ]:
from pinecone import RerankModel
reranked = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=query,
    documents=[{"id": str(i), "text": doc} for i, doc in enumerate(documents)],
    top_n=3 # e.g., 3
)

#### 6. Inspect reranked results

Replace `...` with code that prints out the rank (`i+1`), the similarity score `m.score`, and the document text `m.document.text`. Also fill in the correct attribute for `reranked`. Seeing these values demonstrates how the reranker orders documents and what scores it assigns.

In [ ]:
def show_reranked_results(query, matches):
    print(f"Query: {query}")
    for i, m in enumerate(matches):
        print(f"{str(i+1).rjust(4)}. Score: {m.score:.4f}, Document: {m.document.text}") # Print the position (i+1), m.score, and m.document.text

show_reranked_results(query, reranked.results) # Fill in the correct attribute

### Part 2: Setup a Serverless Index for Medical Notes

#### 1. Install data & model libraries

Run this installation command in a notebook cell. You’ll use these libraries to load, embed, and manipulate medical note data.

In [ ]:
!pip install pandas torch transformers

#### 2. Import modules & define environment settings

Fill in the cloud provider (like ‘aws’), region (like ‘us-east-1’), and choose an index name. You’re configuring a serverless index tailored to your resource requirements and connecting the client in the proper cloud region.

In [ ]:
import os
import time
import pandas as pd
from pinecone import Pinecone, ServerlessSpec
from transformers import AutoTokenizer, AutoModel
import torch

# Get cloud and region settings (these are defaults that work for most users)
cloud = os.getenv('PINECONE_CLOUD', 'aws') # e.g., 'aws'
region = os.getenv('PINECONE_REGION', 'us-east-1') # e.g., 'us-east-1'

# Define serverless specifications
spec = ServerlessSpec(cloud=cloud, region=region)

# Define index name
index_name = 'medical-notes-index' # Give your index a name

#### 3. Create or recreate the index

Fill in the dimension (384 for our model) and choose a metric (‘cosine’ is recommended). The index’s dimension must match the embedding vectors you’ll insert, otherwise upserts will fail.

In [ ]:
# Clean up any existing index with the same name
if pc.has_index(name=index_name):
    pc.delete_index(name=index_name)

# Create a new index
pc.create_index(
    name=index_name,
    dimension=384, # This matches our embedding model size
    metric='cosine', # Distance metric for similarity
    spec=spec
)

# Wait for the index to be ready
while not pc.describe_index(index_name).status['ready']:
    time.sleep(1)

### Part 3: Load the Sample Data

#### 1. Download & read JSONL

Insert the raw GitHub URL: `https://raw.githubusercontent.com/pinecone-io/examples/refs/heads/master/docs/data/sample_notes_data.jsonl`. Downloads sample medical notes data that’s already been processed and embedded for you.

In [ ]:
import requests
import tempfile

with tempfile.TemporaryDirectory() as tmpdirname:
    file_path = os.path.join(tmpdirname, "sample_notes_data.jsonl")

    # Download the file from github
    url = "https://raw.githubusercontent.com/pinecone-io/examples/refs/heads/master/docs/data/sample_notes_data.jsonl" # Insert the GitHub raw URL here
    response = requests.get(url)
    response.raise_for_status()

    with open(file_path, "wb") as f:
        f.write(response.content)

    df = pd.read_json(file_path, orient='records', lines=True)

#### 2. Preview the DataFrame

Fill in the correct pandas attribute to show DataFrame dimensions. Ensures you have the right columns (e.g., `id`, `values`, `metadata`) before upserting.

In [ ]:
# Show head of the DataFrame
print("Data shape:", df.shape) # Show number of rows and columns
display(df.head())

### Part 4: Upsert Data into the Index

#### 1. Instantiate index client & upsert

Pass the DataFrame variable to the upsert function. This pushes all your note embeddings and metadata into Pinecone for later queries.

In [ ]:
# Instantiate an index client
index = pc.Index(name=index_name)

# Upsert data into index from DataFrame
index.upsert_from_dataframe(df) # Pass the DataFrame

#### 2. Wait for availability

Fill in the condition - what number should the vector count be greater than? Ensures that upserted vectors are fully indexed before you attempt to query.

In [ ]:
def is_fresh(index):
    stats = index.describe_index_stats()
    vector_count = stats.total_vector_count
    print(f"Vector count: {vector_count}")
    return vector_count > 0 # What should this be?

while not is_fresh(index):
    time.sleep(5)

print("Index ready!")
display(index.describe_index_stats())

### Part 5: Query & Embedding Function

#### 1. Define your embedding function

Fill in which dimension to average over (0 or 1). Converts incoming queries into the same vector space as your indexed notes.

In [ ]:
def get_embedding(input_question):
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    encoded_input = tokenizer(input_question, padding=True, truncation=True, return_tensors='pt')
    with torch.no_grad():
        model_output = model(**encoded_input)
        embedding = model_output.last_hidden_state[0].mean(dim=1) # Which dimension to average?
    return embedding

#### 2. Run a semantic search query

Write a medical question and choose how many results to retrieve. Retrieves the most semantically similar notes from the index based on your clinical query.

In [ ]:
# Build a query to search
question = "What is the recommended treatment for a patient experiencing severe chest pain?" # Ask a medical question
query_vector = get_embedding(question).tolist()

# Get results
results = index.query(vector=query_vector, top_k=5, include_metadata=True) # Changed vector to query_vector to match variable name

# Sort results by score in descending order
sorted_matches = sorted(results['matches'], key=lambda x: x['score'], reverse=True)

### Part 6: Display & Rerank Clinical Notes

#### 1. Display initial search results

Fill in the correct dictionary keys for score and metadata. Helps you see which notes were initially considered most relevant.

In [ ]:
def show_results(question, matches):
    print(f'Question: \'{question}\'')
    print('\nResults:')
    for i, match in enumerate(matches):
        print(f'{str(i+1).rjust(4)}. ID: {match["id"]}')
        print(f' Score: {match["score"]:.4f}') # What field contains the score?
        print(f' Metadata: {match["metadata"]}') # What field contains metadata?
        print('')

show_results(question, sorted_matches)

#### 2. Prepare documents for reranking

Constructs a field summarizing each note’s metadata for the reranker to use when rescoring.

In [ ]:
# Create documents with concatenated metadata field as "reranking_field" field
transformed_documents = [
    {
        'id': match['id'],
        'reranking_field': '; '.join([f"{key}: {value}" for key, value in match['metadata'].items()])
    }
    for match in results['matches']
]

#### 3. Execute serverless reranking

Create a more specific medical query and choose how many reranked results to return. Reranking uses the refined query and metadata field to reorder notes by their new relevance scores.

In [ ]:
# Define a more specific query for reranking
refined_query = "A patient presents with severe chest pain, shortness of breath, and radiating pain to the left arm. Evaluate for myocardial infarction." # Make a more specific medical question

# Perform reranking based on the query and specified field
reranked_results = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=refined_query,
    documents=transformed_documents,
    rank_fields=["reranking_field"],
    top_n=3, # How many top results do you want?
    return_documents=True,
)

#### 4. Show reranked results

Fill in the attributes for the reranking score, searchable field, and results collection. Allows you to compare how the reranker improves result ordering against the original search.

In [ ]:
def show_reranked_results(question, matches):
    print(f'Question: \'{question}\'')
    print('\nReranked Results:')
    for i, match in enumerate(matches):
        print(f'{str(i+1).rjust(4)}. ID: {match.document.id}')
        print(f' Score: {match.score:.4f}') # What attribute contains the reranking score?
        print(f' Reranking Field: {match.document.reranking_field}') # What contains the searchable field?
        print('')
show_reranked_results(refined_query, reranked_results.results) # What attribute contains the results?

#### 5. Clean up (optional)

Run this when you’re done to avoid unnecessary charges. Serverless indexes cost money when they contain data, so clean up after experiments.

In [ ]:
# Delete the index to save resources
pc.delete_index(name=index_name)